#### !pwd

## cAvatar architecture with cAvatar dataset starts from here

## Dataset read and prepare

In [22]:
import os
import cv2
import pickle
import numpy as np
import random
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
import math
from torchvision import models
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json
from smpl_torch import SMPLModel  # Import the SMPL model from your saved file
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# # Define parameters for a smaller dataset
window_size = 20  # Reduced window size for testing

In [23]:
folder_path = '/x1/mkeya/cAvatar_dataset/touch_normalized.p'

with open(folder_path, 'rb') as f:
        data = pickle.load(f)  # Load the entire .p file
        

In [24]:
print(f"data type: {type(data)}")

print(f"data shape: {data.shape}")

# frames = data

data type: <class 'numpy.ndarray'>
data shape: (11075, 96, 96)


In [25]:
missing_frames_file = '/x1/mkeya/VIBE/missing_frames.txt'

In [26]:
# Read missing frames
with open(missing_frames_file, 'r') as f:
    missing_frames = set(int(line.strip()) for line in f)

# Filter out missing frames from the data
filtered_data = [frame for i, frame in enumerate(data) if i not in missing_frames]

In [27]:
print(f"filtered_data type: {type(filtered_data)}")

filtered_data_array = np.array(filtered_data)

print(f"filtered_data_array shape: {filtered_data_array.shape}")

filtered_data type: <class 'list'>
filtered_data_array shape: (4422, 96, 96)


In [28]:
# print(f"original carpet data shape: {data[63]}")

# print(f"filtered_data_array: {filtered_data_array[63]}")

In [29]:
frames = filtered_data_array

len(frames)

4422

## Teacher Dataset

In [30]:


# Path to the JSON file saved by demo.py
json_file_path = '/x1/mkeya/VIBE/merged_vibe_results.json'  # Replace with the actual path to the JSON file

# Load the JSON file with ground truth values
with open(json_file_path, 'r') as f:
    vibe_results = json.load(f)
print('---------------')



ground_truth_pose = torch.tensor(vibe_results['pose'], dtype=torch.float64)
ground_truth_shape = torch.tensor(vibe_results['shape'], dtype=torch.float64)
ground_truth_camera = torch.tensor(vibe_results['camera'], dtype=torch.float64)
ground_truth_joints = torch.tensor(vibe_results['joints'], dtype=torch.float64)
ground_truth_vertices = torch.tensor(vibe_results['vertices'], dtype=torch.float64)

# taking COCO joints, last 24
ground_truth_joints_coco = ground_truth_joints[:, 25:, :]

print(f"ground_truth_pose shape: {ground_truth_pose.shape}") #theta
print(f"ground_truth_shape shape: {ground_truth_shape.shape}") #beta
print(f"ground_truth_camera shape: {ground_truth_camera.shape}") #translation
# print(f"ground_truth_joints shape: {ground_truth_joints.shape}")
print(f"ground_truth_vertices shape: {ground_truth_vertices.shape}")

print(f"ground_truth_joints_coco shape: {ground_truth_joints_coco.shape}") # joints




---------------
ground_truth_pose shape: torch.Size([5158, 72])
ground_truth_shape shape: torch.Size([5158, 10])
ground_truth_camera shape: torch.Size([5158, 3])
ground_truth_vertices shape: torch.Size([5158, 6890, 3])
ground_truth_joints_coco shape: torch.Size([5158, 24, 3])


## Create Training input and target pair

## Sliding window + Splitting the dataset into training, validation and testing set

In [31]:
def extract_from_list(key, sequences): 
    # key is a string for key name. 
    # Sequences is a list (each element is a small directionary)
    sequences_list = [item[key] for item in sequences]
    return sequences_list



# Step 3: Create overlapping sequences using a reduced sliding window
sequences = []
for i in range(len(frames) - window_size + 1):
    sequence = frames[i:i + window_size]  # Get 20 frames
    #sequences.append(np.array(sequence))  # Shape (20, 96, 96)

    samples_X_y = {} # it's very important to make it empty here.
    keys = ['sequence', 'pose', 'shape', 'camera', 'vertices', 'coco']
    samples_X_y = dict.fromkeys(keys)
    
    # the following data is the learning target for 20th frame from VIBE output
    # in python a[i] is the (i+1)th element in list a

    samples_X_y['sequence']=np.array(sequence)
    samples_X_y['pose']=ground_truth_pose[i+window_size-1:i+window_size]
    samples_X_y['shape']=ground_truth_shape[i+window_size-1:i+window_size]
    samples_X_y['camera']=ground_truth_camera[i+window_size-1:i+window_size]
    samples_X_y['vertices']=ground_truth_vertices[i+window_size-1:i+window_size]
    samples_X_y['coco']=ground_truth_joints_coco[i+window_size-1:i+window_size]

    sequences.append(samples_X_y) 

# # Step: Shuffle the dataset 
# random.seed(42)
# random.shuffle(sequences)

# Step 4: Split sequences into training, validation, and testing sets (80-10-10 split)
num_sequences = len(sequences)
print(f"num_sequences: {num_sequences}")
train_end = int(num_sequences * 0.8)
val_end = train_end + int(num_sequences * 0.1)

# you are not randomizing the samples, just following the original orders of the continues frames
# each sample in the list, is a dictionary
train_sequences1 = sequences[:train_end]

print(f"train_sequences1 shape: {len(train_sequences1)}")
# print(f"train_sequences1 [0]: {train_sequences1[0]}")
train_sequences = extract_from_list('sequence', train_sequences1)

print(f"train_sequences len: {len(train_sequences)}")

val_sequences1 = sequences[train_end:val_end]
val_sequences = extract_from_list('sequence', val_sequences1)

test_sequences1 = sequences[val_end:]
test_sequences = extract_from_list('sequence', test_sequences1)


# Dynamic extraction using globals()
VIBE_keys = ['pose', 'shape', 'camera', 'vertices', 'coco']
name_list = ['ground_truth_pose1','ground_truth_shape1','ground_truth_camera1','ground_truth_vertices1','ground_truth_coco1']

for index, key in enumerate(VIBE_keys):
    # print(index)
    # print(key)
    name1 = name_list[index]
    name2 = name1 

    # print(name2)
    globals()[name2] = extract_from_list(key, train_sequences1)

new_name_list = [item+'_train' for item in name_list]
# Step 5: Print shapes to verify
print(f"Training set shape: {np.array(train_sequences).shape}") 
print(f"Validation set shape: {np.array(val_sequences).shape}")
print(f"Testing set shape: {np.array(test_sequences).shape}")

# 4. Extract ground truth for training, validation, and testing
ground_truth_pose_train = extract_from_list('pose', train_sequences1)
ground_truth_pose_val = extract_from_list('pose', val_sequences1)
ground_truth_pose_test = extract_from_list('pose', test_sequences1)

ground_truth_shape_train = extract_from_list('shape', train_sequences1)
ground_truth_shape_val = extract_from_list('shape', val_sequences1)
ground_truth_shape_test = extract_from_list('shape', test_sequences1)

ground_truth_camera_train = extract_from_list('camera', train_sequences1)
ground_truth_camera_val = extract_from_list('camera', val_sequences1)
ground_truth_camera_test = extract_from_list('camera', test_sequences1)

ground_truth_vertices_train = extract_from_list('vertices', train_sequences1)
ground_truth_vertices_val = extract_from_list('vertices', val_sequences1)
ground_truth_vertices_test = extract_from_list('vertices', test_sequences1)

ground_truth_coco_train = extract_from_list('coco', train_sequences1)
ground_truth_coco_val = extract_from_list('coco', val_sequences1)
ground_truth_coco_test = extract_from_list('coco', test_sequences1)

# Convert to tensors if needed
ground_truth_pose_train = torch.tensor(np.vstack(ground_truth_pose_train))
ground_truth_pose_val = torch.tensor(np.vstack(ground_truth_pose_val))
ground_truth_pose_test = torch.tensor(np.vstack(ground_truth_pose_test))

# print(f"ground_truth_pose_train shape: {ground_truth_pose_train.shape}")
# print(f"ground_truth_pose_val shape: {ground_truth_pose_val.shape}")
print(f"ground_truth_pose_test shape: {ground_truth_pose_test.shape}")

ground_truth_shape_train = torch.tensor(np.vstack(ground_truth_shape_train))
ground_truth_shape_val = torch.tensor(np.vstack(ground_truth_shape_val))
ground_truth_shape_test = torch.tensor(np.vstack(ground_truth_shape_test))

# print(f"ground_truth_shape_train shape: {ground_truth_shape_train.shape}")
# print(f"ground_truth_shape_val shape: {ground_truth_shape_val.shape}")
# print(f"ground_truth_shape_test shape: {ground_truth_shape_test.shape}")

ground_truth_camera_train = torch.tensor(np.vstack(ground_truth_camera_train))
ground_truth_camera_val = torch.tensor(np.vstack(ground_truth_camera_val))
ground_truth_camera_test = torch.tensor(np.vstack(ground_truth_camera_test))

# print(f"ground_truth_camera_train shape: {ground_truth_camera_train.shape}")
# print(f"ground_truth_camera_val shape: {ground_truth_camera_val.shape}")
# print(f"ground_truth_camera_test shape: {ground_truth_camera_test.shape}")

ground_truth_vertices_train = torch.tensor(np.vstack(ground_truth_vertices_train))
ground_truth_vertices_val = torch.tensor(np.vstack(ground_truth_vertices_val))
ground_truth_vertices_test = torch.tensor(np.vstack(ground_truth_vertices_test))

# print(f"ground_truth_vertices_train shape: {ground_truth_vertices_train.shape}")
# print(f"ground_truth_vertices_val shape: {ground_truth_vertices_val.shape}")
# print(f"ground_truth_vertices_test shape: {ground_truth_vertices_test.shape}")

ground_truth_coco_train = torch.tensor(np.vstack(ground_truth_coco_train))
ground_truth_coco_val = torch.tensor(np.vstack(ground_truth_coco_val))
ground_truth_coco_test = torch.tensor(np.vstack(ground_truth_coco_test))

# print(f"ground_truth_coco_train shape: {ground_truth_coco_train.shape}")
# print(f"ground_truth_coco_val shape: {ground_truth_coco_val.shape}")
# print(f"ground_truth_coco_test shape: {ground_truth_coco_test.shape}")

# ground_truth_pose1 = torch.tensor(np.vstack(ground_truth_pose1)) 
# ground_truth_shape1 = torch.tensor(np.vstack(ground_truth_shape1))
# ground_truth_camera1 = torch.tensor(np.vstack(ground_truth_camera1))
# ground_truth_vertices1 = torch.tensor(np.vstack(ground_truth_vertices1))
# ground_truth_coco1 = torch.tensor(np.vstack(ground_truth_coco1))


# print(ground_truth_pose1.shape)
# print(ground_truth_shape1.shape)
# print(ground_truth_camera1.shape)
# print(ground_truth_vertices1.shape)
# print(ground_truth_coco1.shape) # joints

# print(f"ground_truth_coco_train shape: {ground_truth_coco_train.shape}")

# train_sequences_tensor = torch.tensor(train_sequences)

num_sequences: 4403
train_sequences1 shape: 3522
train_sequences len: 3522
Training set shape: (3522, 20, 96, 96)
Validation set shape: (440, 20, 96, 96)
Testing set shape: (441, 20, 96, 96)
ground_truth_pose_test shape: torch.Size([441, 72])


In [32]:
print(f"ground_truth_coco_train shape: {ground_truth_coco_train.shape}")

ground_truth_coco_train shape: torch.Size([3522, 24, 3])


## Model initialization

## Student Model

In [33]:



class CustomResNet18(nn.Module):
    def __init__(self, output_dim):
        super(CustomResNet18, self).__init__()
        
        # Load pretrained ResNet18
        resnet = models.resnet18(pretrained=True)
        
        # Modify first conv layer for single channel input
        self.conv1 = nn.Conv2d(in_channels=1, 
                              out_channels=64, 
                              kernel_size=(7, 7), 
                              stride=(2, 2), 
                              padding=(3, 3), 
                              bias=False)
        
        # Initialize with pretrained weights but adapt for single channel
        with torch.no_grad():
            self.conv1.weight.data = torch.mean(resnet.conv1.weight.data, dim=1, keepdim=True)
        
        # Take remaining layers from pretrained model
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        # Add batch normalization
        self.bn_out = nn.BatchNorm2d(512)
        
        # Add dropout for regularization
        self.dropout = nn.Dropout(0.5)
        
        # Multiple fully connected layers with residual connections
        self.fc1 = nn.Linear(512, 1024)
        self.fc_bn1 = nn.BatchNorm1d(1024)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc_bn2 = nn.BatchNorm1d(1024)
        self.fc3 = nn.Linear(1024, output_dim)
        
        # Residual connection
        self.fc_residual = nn.Linear(512, output_dim)
        
        # Initialize new layers
        nn.init.kaiming_normal_(self.fc1.weight)
        nn.init.kaiming_normal_(self.fc2.weight)
        nn.init.kaiming_normal_(self.fc3.weight)
        nn.init.kaiming_normal_(self.fc_residual.weight)

    def forward(self, x):
        # ResNet backbone
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Additional batch norm
        x = self.bn_out(x)
        
        # Global average pooling
        x = torch.mean(x, dim=(2, 3))  # (batch_size, 512)
        
        # Store residual connection
        residual = self.fc_residual(x)
        
        # Main branch with multiple FC layers
        x = self.fc1(x)
        x = self.fc_bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = self.fc_bn2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.fc3(x)
        
        # Add residual connection
        x = x + residual
        
        return x




def positional_encoding(seq_len, d_model):
    """
    Generates sinusoidal positional encoding matrix of shape (seq_len, d_model).
    """
    position = torch.arange(0, seq_len).unsqueeze(1).float()  # Shape: (seq_len, 1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # Shape: (seq_len, d_model)

# Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super(TransformerEncoderLayer, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)

    def forward(self, src):
        src2 = self.norm1(src)
        attn_output, _ = self.self_attn(src2, src2, src2)
        src = src + self.dropout(attn_output)
        src2 = self.norm2(src)
        src = src + self.dropout(self.linear2(F.relu(self.linear1(src2))))
        return src

class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, nhead, dim_feedforward, dropout):
        super(TransformerEncoder, self).__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, src):
        for layer in self.layers:
            src = layer(src)
        return src

# Usage in StudentModel
class StudentModel(nn.Module):
    def __init__(self, output_dim=86, num_layers=6, d_model=512, nhead=8, 
                 dim_feedforward=2048, dropout=0.1, max_seq_len=20):
        super(StudentModel, self).__init__()
        
        # Initialize with larger output dimension for feature extraction
        self.resnet_model = CustomResNet18(d_model)
        
        # Add layer normalization
        self.layer_norm = nn.LayerNorm(d_model)
        
        # Rest of the StudentModel remains the same
        self.positional_encoding = positional_encoding(seq_len=max_seq_len, d_model=d_model)
        self.transformer_encoder = TransformerEncoder(num_layers, d_model, nhead, 
                                                    dim_feedforward, dropout)
        self.output_layer = nn.Linear(d_model, output_dim)
        self.fc_layer = nn.Linear(d_model, d_model)



# The main Student Model that includes all components
#class StudentModel(nn.Module):
#    def __init__(self, output_dim=86, num_layers=6, d_model=256, nhead=4, dim_feedforward=2048, dropout=0.1, max_seq_len=20):
#        super(StudentModel, self).__init__()
#        self.resnet_model = CustomResNet18(d_model)
#        self.positional_encoding = positional_encoding(seq_len=max_seq_len, d_model=d_model)
#        self.transformer_encoder = TransformerEncoder(num_layers, d_model, nhead, dim_feedforward, dropout)
#        self.output_layer = nn.Linear(d_model, output_dim)  # Output layer to generate joint angles, shapes, etc.
#        self.fc_layer = nn.Linear(in_features=d_model, out_features=d_model)  # Connect ResNet18 and Transformer

    def forward(self, x):
        # Step 1: ResNet for each frame in the sequence
        # x is of shape (batch_size, seq_len, 96, 96), reshape to (batch_size * seq_len, 1, 96, 96)
        batch_size, seq_len, h, w = x.shape
        x_reshaped = x.view(batch_size * seq_len, 1, h, w)
        resnet_output = self.resnet_model(x_reshaped)  # Output shape: (batch_size * seq_len, 256)
        # print(f"resnet_output shape before: {resnet_output.shape}")
        
        # Reshape the output back to (batch_size, seq_len, 256)
        resnet_output = resnet_output.view(batch_size, seq_len, -1)
        # print(f"resnet_output shape after: {resnet_output.shape}")

        # Step 2: Linear layer after ResNet to match the transformer input size
        final_output = self.fc_layer(resnet_output)  # Shape: (batch_size, seq_len, 256)
        # print(f"final_output from step2: {final_output.shape}")

        # Step 3: Add positional encoding
        batch_size, seq_len, d_model = final_output.shape
        positional_encoding = self.positional_encoding[:seq_len, :].unsqueeze(0).repeat(batch_size, 1, 1)  # Shape: (batch_size, seq_len, d_model)
        # print(f"positional_encoding shape: {positional_encoding.shape}")
        
        final_output_with_pe = final_output + positional_encoding  # Add positional encoding to the final output

        # Step 4: Transformer Encoder
        transformer_output = self.transformer_encoder(final_output_with_pe)  # Shape: (batch_size, seq_len, d_model)

        # Step 5: Use the last time step's embedding to predict the 21st frame
        last_embedding = transformer_output[:, -1, :]  # Shape: (batch_size, d_model)
        output_21st_frame = self.output_layer(last_embedding)  # Shape: (batch_size, output_dim)
        # print(f"output_21st_frame shape: {output_21st_frame.shape}")

        # Step 6: Separate Beta, Theta, and Translation
        beta_student = output_21st_frame[..., :10]
        theta_student = output_21st_frame[..., 10:82]
        translation_student = output_21st_frame[..., 82:85]

        return beta_student, theta_student, translation_student


## Initiate training

In [34]:
# import torch
# print(torch.version.cuda)  # Confirm CUDA version
# print(torch.cuda.is_available())  # Should return True

# import torch
# print(torch.cuda.is_available())
# print(torch.cuda.current_device)

import torchvision
print(torchvision.__version__)


# import numpy as np
# print(np.__version__)  # Verify numpy version

0.14.1


In [35]:

# Mean Absolute Error loss function
mae_loss = nn.L1Loss()

# Instantiate the model
student_model = StudentModel()

# Move model to device (e.g., GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# device = torch.device("cuda")

student_model = student_model.to(device)

/x1/mkeya/.conda/envs/vibe_env/lib/python3.7/site-packages/torchvision/models/_utils.py:209: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  f"The parameter '{pretrained_param}' is deprecated since 0.13 and may be removed in the future, "
/x1/mkeya/.conda/envs/vibe_env/lib/python3.7/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [36]:
# Assuming train_sequences is already a tensor
train_sequences = torch.tensor(train_sequences, dtype=torch.float32) # (num_windows, window_size, height, width) = 7299, 20, 96, 96
# train_targets = torch.tensor(reshaped_targets, dtype=torch.float32)
test_sequences = torch.tensor(test_sequences, dtype=torch.float32)

batch_size = 32
num_samples = train_sequences.shape[0]

print(f"train_sequences shape: {train_sequences.shape}")
print(f"test_sequences shape: {test_sequences.shape}")

print(f"num_samples: {num_samples}")

train_sequences shape: torch.Size([3522, 20, 96, 96])
test_sequences shape: torch.Size([441, 20, 96, 96])
num_samples: 3522


In [37]:
# ... existing code ...

def batch_rodrigues(axisang):
    # This function is borrowed from https://github.com/MandyMo/pytorch_HMR/blob/master/src/util.py#L37
    # axisang N x 3
    axisang_norm = torch.norm(axisang + 1e-8, p=2, dim=1)
    angle = torch.unsqueeze(axisang_norm, -1)
    axisang_normalized = torch.div(axisang, angle)
    angle = angle * 0.5
    v_cos = torch.cos(angle)
    v_sin = torch.sin(angle)
    quat = torch.cat([v_cos, v_sin * axisang_normalized], dim=1)
    rot_mat = quat2mat(quat)
    rot_mat = rot_mat.view(rot_mat.shape[0], 9)
    return rot_mat


class CustomLoss(nn.Module):
    def __init__(self, device='cuda'):
        super(CustomLoss, self).__init__()
        self.device = device
        self.criterion_shape = nn.L1Loss().to(self.device)
        self.criterion_keypoints = nn.MSELoss(reduction='none').to(self.device)
        self.criterion_regr = nn.MSELoss().to(self.device)
        
        # Adjusted loss weights
        self.e_pose_loss_weight = torch.tensor(60.0, dtype=torch.float32).to(device)
        self.e_shape_loss_weight = torch.tensor(0.1, dtype=torch.float32).to(device)
        self.e_3d_loss_weight = torch.tensor(30.0, dtype=torch.float32).to(device)

    def keypoint_3d_loss(self, pred_keypoints_3d, gt_keypoints_3d):
        """Modified 3D keypoint loss with better normalization"""
        if len(gt_keypoints_3d) > 0:
            # Center the keypoints at the root joint
            gt_pelvis = gt_keypoints_3d[:, 0,:]
            gt_keypoints_3d = gt_keypoints_3d - gt_pelvis[:, None, :]
            pred_pelvis = pred_keypoints_3d[:, 0,:]
            pred_keypoints_3d = pred_keypoints_3d - pred_pelvis[:, None, :]
            
            # Apply L2 loss with per-joint confidence weighting
            error = ((pred_keypoints_3d - gt_keypoints_3d) ** 2).sum(dim=-1)
            return error.mean()
        else:
            return torch.FloatTensor(1).fill_(0.).to(self.device)

    def smpl_losses(self, pred_pose, pred_shape, gt_pose, gt_shape):
        """Modified SMPL parameter loss with better rotation handling"""
        # Convert axis-angle to rotation matrices
        pred_rotmat = batch_rodrigues(pred_pose.reshape(-1,3)).reshape(-1, 24, 3, 3)
        gt_rotmat = batch_rodrigues(gt_pose.reshape(-1,3)).reshape(-1, 24, 3, 3)
        
        if len(pred_rotmat) > 0:
            # Geodesic loss for rotations
            loss_pose = self.compute_geodesic_loss(pred_rotmat, gt_rotmat)
            # L2 loss for shape parameters
            loss_shape = self.criterion_regr(pred_shape, gt_shape)
        else:
            loss_pose = torch.FloatTensor(1).fill_(0.).to(self.device)
            loss_shape = torch.FloatTensor(1).fill_(0.).to(self.device)
            
        return loss_pose, loss_shape

    def compute_geodesic_loss(self, pred_rot, gt_rot):
        """Compute geodesic loss between rotation matrices"""
        batch_size = pred_rot.shape[0]
        R_diff = torch.matmul(pred_rot, gt_rot.transpose(2, 3))
        trace = torch.diagonal(R_diff, dim1=-2, dim2=-1).sum(-1)
        angle = torch.acos(torch.clamp((trace - 1) / 2, -1 + 1e-6, 1 - 1e-6))
        return angle.mean()



## Discriminator

In [38]:
class Discriminator(nn.Module):
    def __init__(self, num_joints=24, hidden_dim=256):
        super(Discriminator, self).__init__()
        self.hidden_layer = nn.Sequential(
            nn.Linear(num_joints * 3, hidden_dim),
            nn.BatchNorm1d(hidden_dim),  # Add batch normalization
            nn.LeakyReLU(0.2),  # Use LeakyReLU
            nn.Dropout(0.3)  # Add dropout
        )
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, joints):
        joints_flat = joints.view(joints.size(0), -1)
        hidden_output = self.hidden_layer(joints_flat)
        output = self.output_layer(hidden_output)
        return output

# Instantiate the discriminator
discriminator = Discriminator(num_joints=24).to(device)

# Discriminator training optimizer
d_optimizer = optim.Adam(discriminator.parameters(), lr=5e-6, weight_decay=1e-5) # reduced lr and added weight decay

print(f"discriminator hidden layer weights dtype: {discriminator.hidden_layer[0].weight.dtype}")

# Define the Binary Cross-Entropy Loss
criterion = nn.BCELoss()

discriminator hidden layer weights dtype: torch.float32


## Test set

In [39]:
def test_model(model, student_model, discriminator, test_sequences, ground_truth_shape_test, 
               ground_truth_pose_test, ground_truth_camera_test, ground_truth_vertices_test, 
               ground_truth_coco_test, batch_start, batch_end, batch_size, lambda_adv=0.05, lambda_ca=1.0):
    """
    Function to test the model and return total loss and discriminator loss for a single batch.
    """
    model.eval()
    student_model.eval()
    discriminator.eval()

    # epoch_loss_test = []
    # discriminator_loss_test = []

    with torch.no_grad():
        # Extract batch of sliding windows for testing
        batch_windows = test_sequences[batch_start:batch_end].to(device)
        
        # Get the ground truth for the current batch
        ground_truth_shape_batch = ground_truth_shape_test[batch_start:batch_end]
        ground_truth_pose_batch = ground_truth_pose_test[batch_start:batch_end]
        ground_truth_camera_batch = ground_truth_camera_test[batch_start:batch_end]
        ground_truth_vertices_batch = ground_truth_vertices_test[batch_start:batch_end]
        ground_truth_joints_batch = ground_truth_coco_test[batch_start:batch_end]

        if batch_windows.shape[0] != batch_size:
            return None, None  # Skip if batch size mismatch

        # Get the beta, theta, and translation from the student model
        beta_student, theta_student, translation_student = student_model(batch_windows)

        # Forward pass through the SMPL model
        all_vertices = []
        all_joints = []

        for b in range(batch_size):
            beta = beta_student[b].to(device).to(torch.float64)
            theta = theta_student[b].to(device).to(torch.float64)
            translation = translation_student[b].to(device).to(torch.float64)
            vertices, joints = model(beta, theta, translation)
            all_vertices.append(vertices.cpu().numpy())
            all_joints.append(joints.cpu().numpy())

        # all_vertices = torch.tensor(np.array(all_vertices), device=device)
        # all_joints = torch.tensor(np.array(all_joints), device=device)

        all_vertices_torch = torch.tensor(all_vertices, device=device)  # Shape: (32, 6890, 3)
        all_joints_torch = torch.tensor(all_joints, device=device) 

        # # Save the vertices from the last batch for plotting
        # # Example joints data (just for illustration, use your actual `all_joints[0]`)
        # frame_joints = ground_truth_joints_batch[0]
        # joints = frame_joints

        # # Visualize and display the plot for the current batch
        # skeleton = [
        # [0, 1], 
        # [0, 2], [0, 3], [1, 4], [2, 5], [3, 6],
        # [4, 7], [5, 8], [6, 9], [7, 10], [8, 11], [9, 12],
        # [9, 13], [9, 14], [12, 15], [13, 16], [14, 17],
        # [16, 18], [17, 19], [18, 20], [19, 21], [20, 22], [21, 23]
        # ]
        
        # fig = plt.figure(figsize=(8, 8))
        
        # ax = fig.add_subplot(111, projection='3d')
        # predicted_joints = all_joints_torch.cpu().numpy()[0]  # Example: using first sample in batch
        # ground_truth_joints = ground_truth_joints_batch.cpu().numpy()[0]  # First sample in batch
        
        # ax.scatter(predicted_joints[:, 0], predicted_joints[:, 1], predicted_joints[:, 2], c='c', label='Predicted')
        # ax.scatter(ground_truth_joints[:, 0], ground_truth_joints[:, 1], ground_truth_joints[:, 2], c='m', label='Ground Truth')
        
        # # Connect joints according to the skeleton structure
        # for connection in skeleton:
        #     joint_start, joint_end = connection
        #     ax.plot([predicted_joints[joint_start, 0], predicted_joints[joint_end, 0]],
        #             [predicted_joints[joint_start, 1], predicted_joints[joint_end, 1]],
        #             [predicted_joints[joint_start, 2], predicted_joints[joint_end, 2]], color='b')
                
        # ax.set_title(f'Epoch {epoch + 1}, Batch {i // batch_size + 1}')
        # ax.set_xlabel('X')
        # ax.set_ylabel('Y')
        # ax.set_zlabel('Z')
        # ax.view_init(elev=90, azim=-90, roll=0)
        # ax.legend()
        
        # # Save frame as an image
        # frame_filename = os.path.join(output_frames_dir, f"epoch_{epoch+1}_batch_{i//batch_size+1}.png")
        # plt.savefig(frame_filename, dpi=300)
        # plt.close(fig)
        
        # # Read the saved image and write it to the video
        # frame = cv2.imread(frame_filename)
        # resized_frame = cv2.resize(frame, frame_size)  # Ensure consistent size
        # video_writer.write(resized_frame)

        # Adversarial loss labels
        real_outputs = discriminator(ground_truth_joints_batch.to(torch.float32))
        fake_outputs = discriminator(all_joints_torch.detach().to(torch.float32))
        d_real_loss = -torch.mean(torch.log(real_outputs + 1e-8))
        d_fake_loss = -torch.mean(torch.log(1 - fake_outputs + 1e-8))
        d_loss = d_real_loss + d_fake_loss

        # discriminator_loss_test.append(d_loss.item())

        # Replace the existing loss calculation with:
        custom_loss = CustomLoss(device=device)

        # In the training loop, update the loss calculation:
        # Calculate individual losses with proper weights
        pose_loss, shape_loss = custom_loss.smpl_losses(theta_student, beta_student, 
                                                    ground_truth_pose_batch, ground_truth_shape_batch)
        joints_loss = custom_loss.keypoint_3d_loss(all_joints_torch, ground_truth_joints_batch)

        # Apply weights from VIBE
        weighted_pose_loss = pose_loss * custom_loss.e_pose_loss_weight
        weighted_shape_loss = shape_loss * custom_loss.e_shape_loss_weight
        weighted_joints_loss = joints_loss * custom_loss.e_3d_loss_weight

        # Compute total loss
        loss_ca = weighted_pose_loss + weighted_shape_loss + weighted_joints_loss



        # Calculate MAE losses
        #beta_loss = mae_loss(beta_student, ground_truth_shape_batch)
        #camera_loss = mae_loss(translation_student, ground_truth_camera_batch)
        #pose_loss = mae_loss(theta_student, ground_truth_pose_batch)
        #vertices_loss = mae_loss(all_vertices_torch, ground_truth_vertices_batch)
        #joints_loss = mae_loss(all_joints_torch, ground_truth_joints_batch)

        # Compute total loss for the batch
        #loss_ca = (torch.mean(beta_loss) + torch.mean(camera_loss) + 
        #           torch.mean(pose_loss) + torch.mean(vertices_loss) + torch.mean(joints_loss))
        
        # adv_loss = -torch.mean(torch.log(discriminator(all_joints_torch.to(torch.float32)) + 1e-8))
        adv_loss = -torch.mean(torch.log(discriminator(all_joints_torch.to(torch.float32)) + 1e-8))  # L_adv
        
        total_loss = lambda_ca * loss_ca + lambda_adv * adv_loss

        return d_loss, total_loss


## VIBE Code

In [40]:
# -*- coding: utf-8 -*-

# Max-Planck-Gesellschaft zur Förderung der Wissenschaften e.V. (MPG) is
# holder of all proprietary rights on this computer program.
# You can only use this computer program if you have closed
# a license agreement with MPG or you get the right to use the computer
# program from someone who is authorized to grant you that right.
# Any use of the computer program without a valid license is prohibited and
# liable to prosecution.
#
# Copyright©2019 Max-Planck-Gesellschaft zur Förderung
# der Wissenschaften e.V. (MPG). acting on behalf of its Max Planck Institute
# for Intelligent Systems. All rights reserved.
#
# Contact: ps-license@tuebingen.mpg.de

import torch
import numpy as np
from torch.nn import functional as F


def batch_rodrigues(axisang):
    # This function is borrowed from https://github.com/MandyMo/pytorch_HMR/blob/master/src/util.py#L37
    # axisang N x 3
    axisang_norm = torch.norm(axisang + 1e-8, p=2, dim=1)
    angle = torch.unsqueeze(axisang_norm, -1)
    axisang_normalized = torch.div(axisang, angle)
    angle = angle * 0.5
    v_cos = torch.cos(angle)
    v_sin = torch.sin(angle)
    quat = torch.cat([v_cos, v_sin * axisang_normalized], dim=1)
    rot_mat = quat2mat(quat)
    rot_mat = rot_mat.view(rot_mat.shape[0], 9)
    return rot_mat


def quat2mat(quat):
    """
    This function is borrowed from https://github.com/MandyMo/pytorch_HMR/blob/master/src/util.py#L50

    Convert quaternion coefficients to rotation matrix.
    Args:
        quat: size = [batch_size, 4] 4 <===>(w, x, y, z)
    Returns:
        Rotation matrix corresponding to the quaternion -- size = [batch_size, 3, 3]
    """
    norm_quat = quat
    norm_quat = norm_quat / norm_quat.norm(p=2, dim=1, keepdim=True)
    w, x, y, z = norm_quat[:, 0], norm_quat[:, 1], norm_quat[:,
                                                             2], norm_quat[:,
                                                                           3]

    batch_size = quat.size(0)

    w2, x2, y2, z2 = w.pow(2), x.pow(2), y.pow(2), z.pow(2)
    wx, wy, wz = w * x, w * y, w * z
    xy, xz, yz = x * y, x * z, y * z

    rotMat = torch.stack([
        w2 + x2 - y2 - z2, 2 * xy - 2 * wz, 2 * wy + 2 * xz, 2 * wz + 2 * xy,
        w2 - x2 + y2 - z2, 2 * yz - 2 * wx, 2 * xz - 2 * wy, 2 * wx + 2 * yz,
        w2 - x2 - y2 + z2
    ],
                         dim=1).view(batch_size, 3, 3)
    return rotMat


def rotation_matrix_to_angle_axis(rotation_matrix):
    """
    This function is borrowed from https://github.com/kornia/kornia

    Convert 3x4 rotation matrix to Rodrigues vector

    Args:
        rotation_matrix (Tensor): rotation matrix.

    Returns:
        Tensor: Rodrigues vector transformation.

    Shape:
        - Input: :math:`(N, 3, 4)`
        - Output: :math:`(N, 3)`

    Example:
        >>> input = torch.rand(2, 3, 4)  # Nx4x4
        >>> output = tgm.rotation_matrix_to_angle_axis(input)  # Nx3
    """
    if rotation_matrix.shape[1:] == (3,3):
        rot_mat = rotation_matrix.reshape(-1, 3, 3)
        hom = torch.tensor([0, 0, 1], dtype=torch.float32,
                           device=rotation_matrix.device).reshape(1, 3, 1).expand(rot_mat.shape[0], -1, -1)
        rotation_matrix = torch.cat([rot_mat, hom], dim=-1)

    quaternion = rotation_matrix_to_quaternion(rotation_matrix)
    aa = quaternion_to_angle_axis(quaternion)
    aa[torch.isnan(aa)] = 0.0
    return aa


def quaternion_to_angle_axis(quaternion: torch.Tensor) -> torch.Tensor:
    """
    This function is borrowed from https://github.com/kornia/kornia

    Convert quaternion vector to angle axis of rotation.

    Adapted from ceres C++ library: ceres-solver/include/ceres/rotation.h

    Args:
        quaternion (torch.Tensor): tensor with quaternions.

    Return:
        torch.Tensor: tensor with angle axis of rotation.

    Shape:
        - Input: :math:`(*, 4)` where `*` means, any number of dimensions
        - Output: :math:`(*, 3)`

    Example:
        >>> quaternion = torch.rand(2, 4)  # Nx4
        >>> angle_axis = tgm.quaternion_to_angle_axis(quaternion)  # Nx3
    """
    if not torch.is_tensor(quaternion):
        raise TypeError("Input type is not a torch.Tensor. Got {}".format(
            type(quaternion)))

    if not quaternion.shape[-1] == 4:
        raise ValueError("Input must be a tensor of shape Nx4 or 4. Got {}"
                         .format(quaternion.shape))
    # unpack input and compute conversion
    q1: torch.Tensor = quaternion[..., 1]
    q2: torch.Tensor = quaternion[..., 2]
    q3: torch.Tensor = quaternion[..., 3]
    sin_squared_theta: torch.Tensor = q1 * q1 + q2 * q2 + q3 * q3

    sin_theta: torch.Tensor = torch.sqrt(sin_squared_theta)
    cos_theta: torch.Tensor = quaternion[..., 0]
    two_theta: torch.Tensor = 2.0 * torch.where(
        cos_theta < 0.0,
        torch.atan2(-sin_theta, -cos_theta),
        torch.atan2(sin_theta, cos_theta))

    k_pos: torch.Tensor = two_theta / sin_theta
    k_neg: torch.Tensor = 2.0 * torch.ones_like(sin_theta)
    k: torch.Tensor = torch.where(sin_squared_theta > 0.0, k_pos, k_neg)

    angle_axis: torch.Tensor = torch.zeros_like(quaternion)[..., :3]
    angle_axis[..., 0] += q1 * k
    angle_axis[..., 1] += q2 * k
    angle_axis[..., 2] += q3 * k
    return angle_axis


def rotation_matrix_to_quaternion(rotation_matrix, eps=1e-6):
    """
    This function is borrowed from https://github.com/kornia/kornia

    Convert 3x4 rotation matrix to 4d quaternion vector

    This algorithm is based on algorithm described in
    https://github.com/KieranWynn/pyquaternion/blob/master/pyquaternion/quaternion.py#L201

    Args:
        rotation_matrix (Tensor): the rotation matrix to convert.

    Return:
        Tensor: the rotation in quaternion

    Shape:
        - Input: :math:`(N, 3, 4)`
        - Output: :math:`(N, 4)`

    Example:
        >>> input = torch.rand(4, 3, 4)  # Nx3x4
        >>> output = tgm.rotation_matrix_to_quaternion(input)  # Nx4
    """
    if not torch.is_tensor(rotation_matrix):
        raise TypeError("Input type is not a torch.Tensor. Got {}".format(
            type(rotation_matrix)))

    if len(rotation_matrix.shape) > 3:
        raise ValueError(
            "Input size must be a three dimensional tensor. Got {}".format(
                rotation_matrix.shape))
    if not rotation_matrix.shape[-2:] == (3, 4):
        raise ValueError(
            "Input size must be a N x 3 x 4  tensor. Got {}".format(
                rotation_matrix.shape))

    rmat_t = torch.transpose(rotation_matrix, 1, 2)

    mask_d2 = rmat_t[:, 2, 2] < eps

    mask_d0_d1 = rmat_t[:, 0, 0] > rmat_t[:, 1, 1]
    mask_d0_nd1 = rmat_t[:, 0, 0] < -rmat_t[:, 1, 1]

    t0 = 1 + rmat_t[:, 0, 0] - rmat_t[:, 1, 1] - rmat_t[:, 2, 2]
    q0 = torch.stack([rmat_t[:, 1, 2] - rmat_t[:, 2, 1],
                      t0, rmat_t[:, 0, 1] + rmat_t[:, 1, 0],
                      rmat_t[:, 2, 0] + rmat_t[:, 0, 2]], -1)
    t0_rep = t0.repeat(4, 1).t()

    t1 = 1 - rmat_t[:, 0, 0] + rmat_t[:, 1, 1] - rmat_t[:, 2, 2]
    q1 = torch.stack([rmat_t[:, 2, 0] - rmat_t[:, 0, 2],
                      rmat_t[:, 0, 1] + rmat_t[:, 1, 0],
                      t1, rmat_t[:, 1, 2] + rmat_t[:, 2, 1]], -1)
    t1_rep = t1.repeat(4, 1).t()

    t2 = 1 - rmat_t[:, 0, 0] - rmat_t[:, 1, 1] + rmat_t[:, 2, 2]
    q2 = torch.stack([rmat_t[:, 0, 1] - rmat_t[:, 1, 0],
                      rmat_t[:, 2, 0] + rmat_t[:, 0, 2],
                      rmat_t[:, 1, 2] + rmat_t[:, 2, 1], t2], -1)
    t2_rep = t2.repeat(4, 1).t()

    t3 = 1 + rmat_t[:, 0, 0] + rmat_t[:, 1, 1] + rmat_t[:, 2, 2]
    q3 = torch.stack([t3, rmat_t[:, 1, 2] - rmat_t[:, 2, 1],
                      rmat_t[:, 2, 0] - rmat_t[:, 0, 2],
                      rmat_t[:, 0, 1] - rmat_t[:, 1, 0]], -1)
    t3_rep = t3.repeat(4, 1).t()

    mask_c0 = mask_d2 * mask_d0_d1
    mask_c1 = mask_d2 * ~mask_d0_d1
    mask_c2 = ~mask_d2 * mask_d0_nd1
    mask_c3 = ~mask_d2 * ~mask_d0_nd1
    mask_c0 = mask_c0.view(-1, 1).type_as(q0)
    mask_c1 = mask_c1.view(-1, 1).type_as(q1)
    mask_c2 = mask_c2.view(-1, 1).type_as(q2)
    mask_c3 = mask_c3.view(-1, 1).type_as(q3)

    q = q0 * mask_c0 + q1 * mask_c1 + q2 * mask_c2 + q3 * mask_c3
    q /= torch.sqrt(t0_rep * mask_c0 + t1_rep * mask_c1 +  # noqa
                    t2_rep * mask_c2 + t3_rep * mask_c3)  # noqa
    q *= 0.5
    return q


def estimate_translation_np(S, joints_2d, joints_conf, focal_length=5000., img_size=224.):
    """
    This function is borrowed from https://github.com/nkolot/SPIN/utils/geometry.py

    Find camera translation that brings 3D joints S closest to 2D the corresponding joints_2d.
    Input:
        S: (25, 3) 3D joint locations
        joints: (25, 3) 2D joint locations and confidence
    Returns:
        (3,) camera translation vector
    """

    num_joints = S.shape[0]
    # focal length
    f = np.array([focal_length,focal_length])
    # optical center
    center = np.array([img_size/2., img_size/2.])

    # transformations
    Z = np.reshape(np.tile(S[:,2],(2,1)).T,-1)
    XY = np.reshape(S[:,0:2],-1)
    O = np.tile(center,num_joints)
    F = np.tile(f,num_joints)
    weight2 = np.reshape(np.tile(np.sqrt(joints_conf),(2,1)).T,-1)

    # least squares
    Q = np.array([F*np.tile(np.array([1,0]),num_joints), F*np.tile(np.array([0,1]),num_joints), O-np.reshape(joints_2d,-1)]).T
    c = (np.reshape(joints_2d,-1)-O)*Z - F*XY

    # weighted least squares
    W = np.diagflat(weight2)
    Q = np.dot(W,Q)
    c = np.dot(W,c)

    # square matrix
    A = np.dot(Q.T,Q)
    b = np.dot(Q.T,c)

    # solution
    trans = np.linalg.solve(A, b)

    return trans


def estimate_translation(S, joints_2d, focal_length=5000., img_size=224.):
    """
    This function is borrowed from https://github.com/nkolot/SPIN/utils/geometry.py

    Find camera translation that brings 3D joints S closest to 2D the corresponding joints_2d.
    Input:
        S: (B, 49, 3) 3D joint locations
        joints: (B, 49, 3) 2D joint locations and confidence
    Returns:
        (B, 3) camera translation vectors
    """

    device = S.device
    # Use only joints 25:49 (GT joints)
    S = S[:, 25:, :].cpu().numpy()
    joints_2d = joints_2d[:, 25:, :].cpu().numpy()
    joints_conf = joints_2d[:, :, -1]
    joints_2d = joints_2d[:, :, :-1]
    trans = np.zeros((S.shape[0], 3), dtype=np.float32)
    # Find the translation for each example in the batch
    for i in range(S.shape[0]):
        S_i = S[i]
        joints_i = joints_2d[i]
        conf_i = joints_conf[i]
        trans[i] = estimate_translation_np(S_i, joints_i, conf_i, focal_length=focal_length, img_size=img_size)
    return torch.from_numpy(trans).to(device)


def rot6d_to_rotmat_spin(x):
    """Convert 6D rotation representation to 3x3 rotation matrix.
    Based on Zhou et al., "On the Continuity of Rotation Representations in Neural Networks", CVPR 2019
    Input:
        (B,6) Batch of 6-D rotation representations
    Output:
        (B,3,3) Batch of corresponding rotation matrices
    """
    x = x.view(-1,3,2)
    a1 = x[:, :, 0]
    a2 = x[:, :, 1]
    b1 = F.normalize(a1)
    b2 = F.normalize(a2 - torch.einsum('bi,bi->b', b1, a2).unsqueeze(-1) * b1)

    # inp = a2 - torch.einsum('bi,bi->b', b1, a2).unsqueeze(-1) * b1
    # denom = inp.pow(2).sum(dim=1).sqrt().unsqueeze(-1) + 1e-8
    # b2 = inp / denom

    b3 = torch.cross(b1, b2)
    return torch.stack((b1, b2, b3), dim=-1)


def rot6d_to_rotmat(x):
    x = x.view(-1,3,2)

    # Normalize the first vector
    b1 = F.normalize(x[:, :, 0], dim=1, eps=1e-6)

    dot_prod = torch.sum(b1 * x[:, :, 1], dim=1, keepdim=True)
    # Compute the second vector by finding the orthogonal complement to it
    b2 = F.normalize(x[:, :, 1] - dot_prod * b1, dim=-1, eps=1e-6)

    # Finish building the basis by taking the cross product
    b3 = torch.cross(b1, b2, dim=1)
    rot_mats = torch.stack([b1, b2, b3], dim=-1)

    return rot_mats

In [48]:

def check_learning_progress(model, batch_windows, ground_truth, epoch):
    """Check if the model is learning by comparing predictions over time"""
    model.eval()
    with torch.no_grad():
        beta_student, theta_student, translation_student = model(batch_windows)
        
        # Check if predictions are changing
        if hasattr(check_learning_progress, 'prev_predictions'):
            beta_diff = torch.mean(torch.abs(beta_student - check_learning_progress.prev_predictions['beta']))
            theta_diff = torch.mean(torch.abs(theta_student - check_learning_progress.prev_predictions['theta']))
            trans_diff = torch.mean(torch.abs(translation_student - check_learning_progress.prev_predictions['trans']))
            
            print(f"\nEpoch {epoch} Learning Check:")
            print(f"Beta change: {beta_diff:.6f}")
            print(f"Theta change: {theta_diff:.6f}")
            print(f"Translation change: {trans_diff:.6f}")
            
            # Check if predictions are stuck
            if beta_diff < 1e-6 and theta_diff < 1e-6 and trans_diff < 1e-6:
                print("WARNING: Model predictions are not changing significantly!")
        
        # Store current predictions
        check_learning_progress.prev_predictions = {
            'beta': beta_student.detach(),
            'theta': theta_student.detach(),
            'trans': translation_student.detach()
        }
    model.train()

class WeightChangeMonitor:
    def __init__(self, model):
        self.model = model
        self.prev_weights = self.get_current_weights()
    
    def get_current_weights(self):
        weights = {}
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                weights[name] = param.data.clone().cpu()
        return weights
    
    def check_weight_updates(self):
        current_weights = self.get_current_weights()
        changes = {}
        
        for name in self.prev_weights.keys():
            if name in current_weights:
                weight_diff = torch.mean(torch.abs(current_weights[name] - self.prev_weights[name]))
                changes[name] = weight_diff.item()
        
        self.prev_weights = current_weights
        return changes

# Initialize weight monitor
weight_monitor = WeightChangeMonitor(student_model)

class LossAnalyzer:
    def __init__(self):
        self.loss_components = {
            'pose_loss': [],
            'shape_loss': [],
            'joints_loss': [],
            'adv_loss': []
        }
    
    def update(self, pose_loss, shape_loss, joints_loss, adv_loss):
        self.loss_components['pose_loss'].append(pose_loss)
        self.loss_components['shape_loss'].append(shape_loss)
        self.loss_components['joints_loss'].append(joints_loss)
        self.loss_components['adv_loss'].append(adv_loss)
    
    def analyze(self):
        analysis = {}
        for name, values in self.loss_components.items():
            if values:
                mean_val = np.mean(values)
                std_val = np.std(values)
                analysis[name] = {
                    'mean': mean_val,
                    'std': std_val,
                    'trend': 'decreasing' if values[-1] < values[0] else 'increasing'
                }
        return analysis

# Initialize loss analyzer
loss_analyzer = LossAnalyzer()

def plot_learning_progress(loss_analyzer):
    """Plot learning progress over time"""
    plt.figure(figsize=(15, 10))
    
    for component, values in loss_analyzer.loss_components.items():
        plt.plot(values, label=component)
    
    plt.title('Loss Components Over Time')
    plt.xlabel('Batch')
    plt.ylabel('Loss Value')
    plt.legend()
    plt.yscale('log')  # Log scale for better visualization
    plt.grid(True)
    plt.show()

class GradientActivationMonitor:
    def __init__(self, model):
        self.model = model
        self.gradients = {}
        self.activations = {}
        self.hooks = []
        self.setup_hooks()
    
    def setup_hooks(self):
        def save_grad(name):
            def hook(grad):
                self.gradients[name] = grad.detach().cpu()
            return hook
        
        def save_activation(name):
            def hook(module, input, output):
                self.activations[name] = output.detach().cpu()
            return hook
        
        # Add hooks for each major component
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, nn.BatchNorm2d)):
                # Register forward hook for activations
                self.hooks.append(
                    module.register_forward_hook(save_activation(name))
                )
                # Register backward hook for gradients
                if hasattr(module, 'weight') and module.weight is not None:
                    self.hooks.append(
                        module.weight.register_hook(save_grad(f"{name}.weight"))
                    )
    
    def clear_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
    
    def get_statistics(self):
        stats = {}
        # Compute statistics for gradients
        for name, grad in self.gradients.items():
            if grad is not None:
                stats[f"grad_{name}_mean"] = grad.mean().item()
                stats[f"grad_{name}_std"] = grad.std().item()
                stats[f"grad_{name}_norm"] = grad.norm().item()
        
        # Compute statistics for activations
        for name, activ in self.activations.items():
            if activ is not None:
                stats[f"activ_{name}_mean"] = activ.mean().item()
                stats[f"activ_{name}_std"] = activ.std().item()
                stats[f"activ_{name}_norm"] = activ.norm().item()
        
        return stats

# Initialize the monitor
monitor = GradientActivationMonitor(student_model)


## Training

In [49]:
# Training parameters
num_windows = train_sequences.shape[0]  # Total sliding windows
print(f"num_windows: {num_windows}")

batch_size = 32  # Batch size for training
num_epochs = 50  # Number of epochs for training
learning_rate = 1.5e-4

# SMPL model
model = SMPLModel(device=device)

# Initialize optimizer 
#optimizer = optim.Adam(student_model.parameters(), lr=learning_rate)

# Modified setup with better learning rate:
#optimizer = optim.Adam(student_model.parameters(), lr=1e-4)  # Start with smaller learning rate

# Modified optimizer and learning rate setup
optimizer = optim.AdamW(student_model.parameters(), 
                       lr=1e-4,  # Lower initial learning rate
                       weight_decay=0.01,  # Increased weight decay for better regularization
                       betas=(0.9, 0.999))

# More aggressive learning rate scheduling
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,  # Reduce LR by half when plateauing
    patience=2,   # Wait 2 epochs before reducing LR
    verbose=True,
    min_lr=1e-6  # Don't let LR go below this
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        if m.bias is not None:
            torch.nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm1d):
        torch.nn.init.constant_(m.weight, 1)
        torch.nn.init.constant_(m.bias, 0)

# Apply initialization
student_model.apply(init_weights)
discriminator.apply(init_weights)

# # Step size scheduler
# step_size = 2  # Reduce learning rate every 2 epochs
# gamma = 0.1  # Multiply learning rate by 0.1
# scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

# collecting losses per epoch
epoch_losses = []
epoch_losses_test = []

# collecting discriminator loss per epoch
discriminator_losses = []
discriminator_losses_test = []

ca_losses = []
adv_losses = []

# saving the output as video
# Create an output directory for saving frames
output_frames_dir = "output_frames_test"
os.makedirs(output_frames_dir, exist_ok=True)

# Video writer setup
video_filename = "test_dsi_g_each_epoch_video.avi"
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Codec
fps = 2  # Frames per second (adjust as needed)
frame_size = (800, 800)  # Ensure consistency with your plot dimensions
video_writer = cv2.VideoWriter(video_filename, fourcc, fps, frame_size)


# Initialize monitoring tools
monitor = GradientActivationMonitor(student_model)
weight_monitor = WeightChangeMonitor(student_model)
loss_analyzer = LossAnalyzer()
print("Monitoring tools initialized")



# Define loss weights
#lambda_ca = 1.0    # Weight for reconstruction loss
#lambda_adv = 0.01  # Weight for adversarial loss

# Adjust loss weights
lambda_ca = 1.0     # Content/reconstruction loss weight
lambda_adv = 0.001  # Very small adversarial weight initially

# Add gradient clipping
max_grad_norm = 1.0
torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_grad_norm)

# Initialize lists to track losses
epoch_losses = []
discriminator_losses = []
total_losses = []
adv_loss_plot = []

for epoch in range(num_epochs):
    epoch_loss = []  # Track losses for this epoch
    discriminator_loss = []
    ca_loss = []
    adver_loss = []

    running_loss = 0.0
    running_pose_loss = 0.0
    running_shape_loss = 0.0
    running_joints_loss = 0.0

    
    for i in range(0, num_windows, batch_size):
        # Extract batch of sliding windows
        batch_windows = train_sequences[i:i + batch_size].to(device).to(torch.float32)  # Ensure float32
        
        # Get the ground truth for the 20th frame of each sliding window in the batch
        ground_truth_shape_batch = ground_truth_shape_train[i:i + batch_size].to(torch.float32)
        ground_truth_pose_batch = ground_truth_pose_train[i:i + batch_size].to(torch.float32)
        ground_truth_camera_batch = ground_truth_camera_train[i:i + batch_size].to(torch.float32)
        ground_truth_vertices_batch = ground_truth_vertices_train[i:i + batch_size].to(torch.float32)
        ground_truth_joints_batch = ground_truth_coco_train[i:i + batch_size].to(torch.float32)

        if batch_windows.shape[0] != batch_size:
            continue

        # Get the beta, theta, and translation from the student model
        beta_student, theta_student, translation_student = student_model(batch_windows)
        
        # Ensure student model outputs are float32
        beta_student = beta_student.to(torch.float32)
        theta_student = theta_student.to(torch.float32)
        translation_student = translation_student.to(torch.float32)

        # Forward pass: SMPL vertices and joints
        all_vertices = []
        all_joints = []

        for b in range(batch_size):

            # Extract beta, theta, and translation for the current frame and convert to float64
            beta = beta_student[b].to(device).to(torch.float64)
            theta = theta_student[b].to(device).to(torch.float64)
            translation = translation_student[b].to(device).to(torch.float64)

            # Forward pass through the SMPL model (which expects float64)
            vertices, joints = model(beta, theta, translation)

            # Convert outputs back to float32 for the rest of the training pipeline
            vertices = vertices.to(torch.float32)
            joints = joints.to(torch.float32)

            

            # Detach tensors before converting to numpy
            all_vertices.append(vertices.detach().cpu().numpy())
            all_joints.append(joints.detach().cpu().numpy())


        # Convert lists to numpy arrays
        all_vertices = np.array(all_vertices, dtype=np.float32)
        all_joints = np.array(all_joints, dtype=np.float32)
        
        # Convert back to torch tensors
        all_vertices_torch = torch.tensor(all_vertices, dtype=torch.float32, device=device)
        all_joints_torch = torch.tensor(all_joints, dtype=torch.float32, device=device)


        # Adversarial loss labels
        real_labels = torch.ones((batch_size, 1), dtype=torch.float32, device=device)
        fake_labels = torch.zeros((batch_size, 1), dtype=torch.float32, device=device)
        
        # unfreeze the discriminator         
        discriminator.train()  # Set discriminator to training mode

        student_model.eval() # Set the student model to eval mode during discriminator training
        for param in discriminator.parameters():
            param.requires_grad = True  # Enable gradients for discriminator
        
        # Forward pass through discriminator
        real_outputs = discriminator(ground_truth_joints_batch.to(torch.float32))  # Real joints
        fake_outputs = discriminator(all_joints_torch.detach().to(torch.float32))  # Generated joints (detach to prevent grad flow)

        # Theoretical discriminator loss (BCE with logits to match formula)
        d_real_loss = -torch.mean(torch.log(real_outputs + 1e-8))  # Avoid log(0)
        d_fake_loss = -torch.mean(torch.log(1 - fake_outputs + 1e-8))  # Avoid log(0)
        d_loss = d_real_loss + d_fake_loss  # Total discriminator loss

        discriminator_loss.append(d_loss.item())
        
        # Backpropagation for discriminator
        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()


        # freeze the discriminator for generator training
        discriminator.eval()
        for param in discriminator.parameters():
            param.requires_grad = False # Freeze the discriminator's weight

        student_model.train() # Set the student model to train mode 
        for param in student_model.parameters():
            param.requires_grad = True # Freeze the discriminator's weight

        # Calculate losses using CustomLoss
        custom_loss = CustomLoss(device=device)
        pose_loss, shape_loss = custom_loss.smpl_losses(theta_student, beta_student, 
                                                       ground_truth_pose_batch, ground_truth_shape_batch)
        joints_loss = custom_loss.keypoint_3d_loss(all_joints_torch, ground_truth_joints_batch)

        # Apply weights from custom loss
        weighted_pose_loss = pose_loss * custom_loss.e_pose_loss_weight
        weighted_shape_loss = shape_loss * custom_loss.e_shape_loss_weight
        weighted_joints_loss = joints_loss * custom_loss.e_3d_loss_weight

        # Compute reconstruction loss
        loss_ca = weighted_pose_loss + weighted_shape_loss + weighted_joints_loss

        # Adversarial loss
        adv_loss = -torch.mean(torch.log(discriminator(all_joints_torch.to(torch.float32)) + 1e-8))

        # Total loss with lambda weights
        total_loss = lambda_ca * loss_ca + lambda_adv * adv_loss

        # Backpropagation
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)

        optimizer.step()


        # Convert tensor losses to float and append for tracking
        # Use .item() only for tensor values
        epoch_loss.append(float(total_loss.item()))
        ca_loss.append(float(loss_ca.item()))
        adver_loss.append(float(adv_loss.item()))

        # For discriminator loss, calculate it here
        real_outputs = discriminator(ground_truth_joints_batch.to(torch.float32))
        fake_outputs = discriminator(all_joints_torch.detach().to(torch.float32))
        d_real_loss = -torch.mean(torch.log(real_outputs + 1e-8))
        d_fake_loss = -torch.mean(torch.log(1 - fake_outputs + 1e-8))
        d_loss = d_real_loss + d_fake_loss
        discriminator_loss.append(float(d_loss.item()))

        # Print batch progress
        if i % (batch_size * 10) == 0:  # Print every 10 batches
            print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i//batch_size}/{num_windows//batch_size}], '
                  f'Loss: {float(total_loss.item()):.4f}, D_Loss: {float(d_loss.item()):.4f}')

        # Calculate individual losses
        pose_loss, shape_loss = custom_loss.smpl_losses(theta_student, beta_student, 
                                                       ground_truth_pose_batch, ground_truth_shape_batch)
        joints_loss = custom_loss.keypoint_3d_loss(all_joints_torch, ground_truth_joints_batch)

        # Track individual losses
        running_pose_loss += pose_loss.item()
        running_shape_loss += shape_loss.item()
        running_joints_loss += joints_loss.item()


        # Monitor learning
        stats = monitor.get_statistics()
        
        # Print statistics every N batches
        if i % (batch_size * 100) == 0:
            print(f"\nBatch {i//batch_size} Statistics:")
            for key, value in stats.items():
                if abs(value) < 1e-8 or abs(value) > 1e8:
                    print(f"WARNING: Potential issue with {key}: {value}")
                print(f"{key}: {value:.6f}")
            
            # Check learning progress
            check_learning_progress(student_model, batch_windows, 
                                  (ground_truth_shape_batch, ground_truth_pose_batch, ground_truth_camera_batch),
                                  epoch)
        
        # Update loss analyzer
        loss_analyzer.update(pose_loss.item(), shape_loss.item(), 
                           joints_loss.item(), adv_loss.item())
    
        
        

        
        # Print detailed progress every N batches
        if i % (batch_size * 10) == 0:
            print(f'Pose Loss: {pose_loss.item():.4f}, Shape Loss: {shape_loss.item():.4f},Joints Loss: {joints_loss.item():.4f},Total Loss: {total_loss.item():.4f}\n')

        # Monitor gradients and activations
        stats = monitor.get_statistics()
        
        # Print important statistics every N batches
        if i % (batch_size * 100) == 0:
            print("\nLayer Statistics:")
            for key, value in stats.items():
                if abs(value) < 1e-8 or abs(value) > 1e8:
                    print(f"WARNING: Potential issue with {key}: {value}")
                print(f"{key}: {value:.6f}")
        
        # Check learning progress
        if i == 0:  # Check first batch of each epoch
            check_learning_progress(student_model, batch_windows, 
                                  (ground_truth_shape_batch, ground_truth_pose_batch, ground_truth_camera_batch),
                                  epoch)


    
    # Print epoch summary
    num_batches = num_windows // batch_size
    print(f'\nEpoch [{epoch+1}/{num_epochs}] Summary:')
    print(f'Average Pose Loss: {running_pose_loss/num_batches:.4f}')
    print(f'Average Shape Loss: {running_shape_loss/num_batches:.4f}')
    print(f'Average Joints Loss: {running_joints_loss/num_batches:.4f}')
    print(f'Average Total Loss: {running_loss/num_batches:.4f}\n')

    avg_loss = running_loss/num_batches
    
    # Step the scheduler with the average loss
    scheduler.step(avg_loss)

    

    

    # Calculate average losses for the epoch
    if len(epoch_loss) > 0:  # Check if we have any losses to average
        avg_loss_per_epoch = sum(epoch_loss) / len(epoch_loss)
        avg_d_loss_per_epoch = sum(discriminator_loss) / len(discriminator_loss)
        avg_ca_loss_per_epoch = sum(ca_loss) / len(ca_loss)
        avg_adv_loss_per_epoch = sum(adver_loss) / len(adver_loss)

        # Store the averages
        epoch_losses.append(avg_loss_per_epoch)
        discriminator_losses.append(avg_d_loss_per_epoch)
        total_losses.append(avg_loss_per_epoch)
        adv_loss_plot.append(avg_adv_loss_per_epoch)

        # Print epoch progress
        print(f'\nEpoch [{epoch+1}/{num_epochs}]')
        print(f'Average Loss: {avg_loss_per_epoch:.4f}')
        print(f'Average D Loss: {avg_d_loss_per_epoch:.4f}')
        print(f'Average CA Loss: {avg_ca_loss_per_epoch:.4f}')
        print(f'Average Adv Loss: {avg_adv_loss_per_epoch:.4f}\n')



    # Update learning rate based on average loss
    avg_loss = sum(epoch_loss) / len(epoch_loss)
    scheduler.step(avg_loss)



    # End of epoch analysis
    print(f"\nEpoch {epoch + 1} Analysis:")
    
    # Check weight updates
    weight_changes = weight_monitor.check_weight_updates()
    print("\nWeight changes after epoch:")
    for name, change in weight_changes.items():
        print(f"{name}: {change:.6f}")
        if change < 1e-7:
            print(f"WARNING: Very small weight update in {name}")
    
    # Analyze loss components
    analysis = loss_analyzer.analyze()
    print("\nLoss Component Analysis:")
    for component, stats in analysis.items():
        print(f"{component}:")
        print(f"  Mean: {stats['mean']:.6f}")
        print(f"  Std: {stats['std']:.6f}")
        print(f"  Trend: {stats['trend']}")
    
    # Clear monitoring data for next epoch
    monitor.gradients.clear()
    monitor.activations.clear()

    # Add at the end of each epoch
    plot_learning_progress(loss_analyzer)



# Clean up
monitor.clear_hooks()

    
    # Filter out 0.0 values
    #filtered_epoch_loss_test = [loss for loss in epoch_loss_test if loss != 0.0]
    #print(f"filtered_epoch_loss_test: {filtered_epoch_loss_test}")
    
    # Calculate average if the filtered list is not empty
    #if filtered_epoch_loss_test:
    #    avg_loss_per_epoch_test = sum(filtered_epoch_loss_test) / len(filtered_epoch_loss_test)
    #else:
    #    avg_loss_per_epoch_test = 0.0  # Handle case when all values are 0.0 or the list is empty
    
    #epoch_losses_test.append(avg_loss_per_epoch_test)

    
    #avg discriminator loss per epoch
    #avg_discriminator_loss_per_epoch = sum(discriminator_loss)/len(discriminator_loss)
    #discriminator_losses.append(avg_discriminator_loss_per_epoch)


    # Filter out 0.0 values
    #filtered_discriminator_loss_test = [loss for loss in discriminator_loss_test if loss != 0.0]
    
    # Calculate average if the filtered list is not empty
    #if filtered_discriminator_loss_test:
    #    avg_discriminator_loss_per_epoch_test = sum(filtered_discriminator_loss_test) / len(filtered_discriminator_loss_test)
    #else:
    #    avg_discriminator_loss_per_epoch_test = 0.0  # Handle case when all values are 0.0 or the list is empty
    
    #discriminator_losses_test.append(avg_discriminator_loss_per_epoch_test)

    #ca_loss_per_epoch = sum(ca_loss)/len(ca_loss)
    #ca_losses.append(ca_loss_per_epoch)

    #adver_loss_per_epoch = sum(adver_loss)/len(adver_loss)
    #adv_losses.append(adver_loss_per_epoch)
    

    # # Step the scheduler after each epoch
    # scheduler.step()
    # print(f"Epoch {epoch + 1}/{num_epochs}, Learning Rate: {scheduler.get_last_lr()[0]}")

# After training, plot the losses
epochs = range(1, num_epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs, total_losses, label="Total Loss", color="blue", marker="o")
plt.plot(epochs, discriminator_losses, label="Discriminator Loss", color="red", marker="o")
plt.plot(epochs, adv_loss_plot, label="Adversarial Loss", color="green", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Losses Over Epochs")
plt.legend()
plt.grid(True)
plt.savefig("training_loss_plot.png")
plt.show()



# Save loss trend to file
with open("loss_log.txt", "w") as f:
    for epoch, (loss, d_loss, loss_test, d_loss_test, adv_l) in enumerate(zip(epoch_losses, discriminator_losses, epoch_losses_test, discriminator_losses_test, adv_losses)):
        d_loss_str = f"{d_loss:.4f}" if d_loss is not None else "N/A"
        f.write(f"Epoch {epoch+1}, Total Loss: {loss:.4f}, Discriminator Loss: {d_loss_str}, Test Loss: {loss_test:.4f}, Test Discrimination Loss: {d_loss_test:.4f}, Adv Loss: {adv_l:.4f}\n")

print("Training complete. Loss log saved to 'loss_log.txt'.")

# Finalize the video after all batches in the epoch
video_writer.release()
print(f"Video saved to {video_filename}")

num_windows: 3522
Monitoring tools initialized
Epoch [1/50], Batch [0/110], Loss: 162.2954, D_Loss: 2.5224

Batch 0 Statistics:
grad_output_layer.weight_mean: -0.006495
grad_output_layer.weight_std: 1.007275
grad_output_layer.weight_norm: 211.366623
grad_transformer_encoder.layers.5.linear2.weight_mean: -0.003210
grad_transformer_encoder.layers.5.linear2.weight_std: 0.129218
grad_transformer_encoder.layers.5.linear2.weight_norm: 132.356689
grad_transformer_encoder.layers.5.linear1.weight_mean: 0.000000
grad_transformer_encoder.layers.5.linear1.weight_std: 0.064318
grad_transformer_encoder.layers.5.linear1.weight_norm: 65.860390
grad_transformer_encoder.layers.5.self_attn.out_proj.weight_mean: 0.000021
grad_transformer_encoder.layers.5.self_attn.out_proj.weight_std: 0.065577
grad_transformer_encoder.layers.5.self_attn.out_proj.weight_norm: 33.574974
grad_transformer_encoder.layers.4.linear2.weight_mean: -0.004487
grad_transformer_encoder.layers.4.linear2.weight_std: 0.136189
grad_transf

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# Initialize lists to store epochs and losses
epochs = []
adv_loss_plot = []
total_losses = []
discriminator_losses = []
test_losses = []
test_discriminator_losses = []

# Read the log file and parse the values
with open("loss_log.txt", "r") as file:
    for line in file:
        parts = line.strip().split(", ")
        if len(parts) == 6:  # Ensure proper formatting
            epoch = int(parts[0].split(" ")[1])  # Extract epoch number
            total_loss = float(parts[1].split(": ")[1])  # Extract Total Loss
            d_loss = float(parts[2].split(": ")[1])  # Extract Discriminator Loss
            test_loss = float(parts[3].split(": ")[1])  # Extract Test Loss
            test_d_loss = float(parts[4].split(": ")[1])  # Extract Test Discrimination Loss
            adv_loss = float(parts[5].split(": ")[1])  # Extract Test Discrimination Loss

            # Append values to lists
            epochs.append(epoch)
            total_losses.append(total_loss)
            discriminator_losses.append(d_loss)
            test_losses.append(test_loss)
            test_discriminator_losses.append(test_d_loss)
            adv_loss_plot.append(adv_loss)

# Plot the losses
plt.figure(figsize=(12, 8))

# Plot total loss
plt.plot(epochs, total_losses, label="Total Loss", color="blue", marker="o")
for i, (x, y) in enumerate(zip(epochs, total_losses)):
    plt.text(x, y, f"{y:.2f}", color="blue", fontsize=9, ha='right')

# Plot discriminator loss
plt.plot(epochs, discriminator_losses, label="Discriminator Loss", color="red", marker="o")
for i, (x, y) in enumerate(zip(epochs, discriminator_losses)):
    plt.text(x, y, f"{y:.2f}", color="red", fontsize=9, ha='left')

# Plot test loss
plt.plot(epochs, test_losses, label="Test Loss", color="green", linestyle="--", marker="x")
for i, (x, y) in enumerate(zip(epochs, test_losses)):
    plt.text(x, y, f"{y:.2f}", color="green", fontsize=9, ha='center')

# Plot test discriminator loss
plt.plot(epochs, test_discriminator_losses, label="Test Discrimination Loss", color="purple", linestyle="--", marker="x")
for i, (x, y) in enumerate(zip(epochs, test_discriminator_losses)):
    plt.text(x, y, f"{y:.2f}", color="purple", fontsize=9, ha='center')

# Plot adv loss
plt.plot(epochs, adv_loss_plot, label="adv Loss", color="black", linestyle="--", marker="x")
for i, (x, y) in enumerate(zip(epochs, adv_loss_plot)):
    plt.text(x, y, f"{y:.2f}", color="purple", fontsize=9, ha='center')



# Add labels, title, and legend
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Losses Over Epochs")
plt.legend()
plt.grid()

# Save the plot as an image (optional)
plt.savefig("loss_plot.png")

# Display the plot
plt.show()


## Model in evaluation mode for testing

In [ ]:
# num_epochs_test = 20
# lambda_adv = 0.01  # Smaller weight for adversarial loss
# lambda_ca = 1.0    # Weight for reconstruction loss

# # Initialize lists to collect loss values
# test_losses = []
# test_discriminator_losses = []

# # Initialize the model and load weights
# model.load_state_dict(torch.load("trained_model.pth"))
# student_model.load_state_dict(torch.load("trained_student_model.pth"))
# discriminator.load_state_dict(torch.load("trained_discriminator.pth"))

# # Set models to evaluation mode
# model.eval()
# student_model.eval()
# discriminator.eval()

# print("Models loaded and set to evaluation mode.")


# # # Evaluation loop (on testing set)
# for epoch in range(num_epochs_test):   
#     epoch_loss_test = []
#     discriminator_loss_test = []

#     with torch.no_grad():  # No gradients will be computed during testing
        
#         for i in range(0, num_windows, batch_size):
            
#             # Extract batch of sliding windows for testing
#             batch_windows = test_sequences[i:i + batch_size].to(device)  # Shape: (batch_size, 20, 96, 96)
    
#             # Get the ground truth for the 20th frame of each sliding window in the batch
#             ground_truth_shape_batch = ground_truth_shape_test[i:i + batch_size]  # Shape: (batch_size, 10)
#             ground_truth_pose_batch = ground_truth_pose_test[i:i + batch_size]    # Shape: (batch_size, 72)
#             ground_truth_camera_batch = ground_truth_camera_test[i:i + batch_size]  # Shape: (batch_size, 3)
#             ground_truth_vertices_batch = ground_truth_vertices_test[i:i + batch_size]  # Shape: (batch_size, 6890, 3)
#             ground_truth_joints_batch = ground_truth_coco_test[i:i + batch_size]  # Shape: (batch_size, 24, 3)
    
#             if batch_windows.shape[0] != batch_size:
#                 continue
    
#             # Get the beta, theta, and translation from the student model (no backprop)
#             beta_student, theta_student, translation_student = student_model(batch_windows)
    
#             # Forward pass: SMPL vertices and joints
#             all_vertices = []
#             all_joints = []
    
#             for b in range(batch_size):
#                 # Extract beta, theta, and translation for the current frame
#                 beta = beta_student[b].to(device).to(torch.float64)  # Shape: (10,)
#                 theta = theta_student[b].to(device).to(torch.float64)  # Shape: (72,)
#                 translation = translation_student[b].to(device).to(torch.float64)  # Shape: (3,)
    
#                 # Forward pass through the SMPL model
#                 vertices, joints = model(beta, theta, translation)  # Shapes: (num_vertices, 3), (24, 3)
    
#                 # Collect vertices and joints for this frame
#                 all_vertices.append(vertices.cpu().numpy())
#                 all_joints.append(joints.cpu().numpy())
    
#             # Convert lists to numpy arrays
#             all_vertices = np.array(all_vertices)  # Shape: (batch_size, num_vertices, 3)
#             all_joints = np.array(all_joints)  # Shape: (batch_size, 24, 3)
    
#             # Convert SMPL output to torch tensors
#             all_vertices_torch = torch.tensor(all_vertices, device=device)  # Shape: (batch_size, 6890, 3)
#             all_joints_torch = torch.tensor(all_joints, device=device)  # Shape: (batch_size, 24, 3)
    
#             # Adversarial loss labels
#             real_labels = torch.ones((batch_size, 1), device=device)  # Real labels (1)
#             fake_labels = torch.zeros((batch_size, 1), device=device)  # Fake labels (0)
            
#             # Forward pass through discriminator
#             real_outputs = discriminator(ground_truth_joints_batch.to(torch.float32))  # Real joints
#             fake_outputs = discriminator(all_joints_torch.detach().to(torch.float32))  # Generated joints (detach to prevent grad flow)
    
#             # Theoretical discriminator loss (BCE with logits to match formula)
#             d_real_loss = -torch.mean(torch.log(real_outputs + 1e-8))  # Avoid log(0)
#             d_fake_loss = -torch.mean(torch.log(1 - fake_outputs + 1e-8))  # Avoid log(0)
#             d_loss = d_real_loss + d_fake_loss  # Total discriminator loss        
    
#             discriminator_loss_test.append(d_loss.item())
    
#             print(f"Epoch {epoch+1}, Batch {i//batch_size+1}, Discriminator Loss: {d_loss.item():.4f}")
    
#             # Calculate the individual MAE losses for each component
#             beta_loss = mae_loss(beta_student, ground_truth_shape_batch)  # Shape: (batch_size, 10)
#             camera_loss = mae_loss(translation_student, ground_truth_camera_batch)  # Shape: (batch_size, 3)
#             pose_loss = mae_loss(theta_student, ground_truth_pose_batch)  # Shape: (batch_size, 72)
#             vertices_loss = mae_loss(all_vertices_torch, ground_truth_vertices_batch)  # Shape: (batch_size, 6890, 3)
#             joints_loss = mae_loss(all_joints_torch, ground_truth_joints_batch)  # Shape: (batch_size, 24, 3)
    
#             # Compute total loss for the evaluation batch
#             loss_ca = (torch.mean(beta_loss) + torch.mean(camera_loss) + 
#                        torch.mean(pose_loss) + torch.mean(vertices_loss) + torch.mean(joints_loss))
    
#             # Adversarial loss for generator
#             adv_loss = -torch.mean(torch.log(discriminator(all_joints_torch.to(torch.float32))))  # L_adv
            
            
#             # Compute total loss
#             total_loss = lambda_ca * loss_ca + lambda_adv * adv_loss
    
#             epoch_loss_test.append(total_loss.item())
    
#             print(f"Testing - Batch {i//batch_size+1}, Total Loss: {total_loss.item():.4f}, Discriminator Loss: {d_loss.item():.4f}")
    
#     #avg total loss per epoch
#     avg_loss_per_epoch = sum(epoch_loss_test)/len(epoch_loss_test)
#     test_losses.append(avg_loss_per_epoch)
#     print(f"Epoch {epoch+1}, Average loss per epoch: {avg_loss_per_epoch:.4f}")

#     #avg discriminator loss per epoch
#     avg_discriminator_loss_per_epoch = sum(discriminator_loss_test)/len(discriminator_loss_test)
#     test_discriminator_losses.append(avg_discriminator_loss_per_epoch)
#     print(f"Epoch {epoch+1}, Average discriminator loss per epoch: {avg_discriminator_loss_per_epoch:.4f}")


# # Optionally, save the test results to a file
# with open("test_loss_log.txt", "w") as f:
#     for epoch, (loss, d_loss) in enumerate(zip(test_losses, test_discriminator_losses)):
#         f.write(f"Batch {epoch+1}, Total Loss: {loss:.4f}, Discriminator Loss: {d_loss:.4f}\n")

# print("Testing complete. Test loss log saved to 'test_loss_log.txt'.")
